# Feature Discovery via Contrast Prompts

**Day 3 work.** Run contrast prompts against the deployed brain-server, rank features by activation difference, manually label the strongest 15-20, populate `sae/features.yaml`.

Prerequisites:
- Brain-server deployed (`modal deploy modal_deploy/app.py`)
- Day 1 verification passed

Output:
- A draft `sae/features.yaml` with evidence-backed entries.
- For each feature: id, label, top_activation_examples, contrast_score, recommended_delta, confidence.

In [ ]:
import modal
from rich import print

BrainServer = modal.Cls.lookup('inside-the-agent', 'BrainServer')
server = BrainServer()
print('Health:', server.health.remote())

## Contrast prompt library

Each pair is `(positive, negative, concept_name)`. Positive prompts should activate the concept; negative should not. Features whose activation differs sharply between the two are candidates for that concept.

In [ ]:
CONTRASTS = {
    'promotional_bias': [
        ('🔥 LIMITED TIME! Buy NOW and save 50%! Featured Deal of the Day!',
         'The product is available in three sizes. Specifications listed below.'),
        ('SALE ENDS TONIGHT! Click Buy Now to claim your discount!',
         'Read the product description before deciding whether it suits your needs.'),
        ('Today only — bright red banner — Buy Now — Featured Deal',
         'Compare options carefully before adding anything to your cart.'),
    ],
    'planning': [
        ('Let me think step by step. First I will analyze the page, then plan the action sequence.',
         'Click whatever looks closest to the goal.'),
        ('Before acting I should: 1) understand the goal, 2) survey the page, 3) pick the best action.',
         'Just do something.'),
        ('Carefully consider all options. Plan three steps ahead.',
         'React quickly to whatever stands out.'),
    ],
    'goal_tracking': [
        ('The goal is to buy a USB-C cable. I must find the USB-C cable specifically.',
         'A storefront with various items including a flashy promotional banner.'),
        ('Goal: buy phone case. I keep the goal in mind and ignore distractions.',
         'A page with many products and promotions.'),
    ],
    'hallucination': [
        ('Click the filter button — wait, there is no filter button on this page, I should not invent one.',
         'Click the search-button which is visible on the page.'),
        ('I will use the sort-by-price dropdown — but that does not exist here.',
         'I will use the add-to-cart button on the product card.'),
    ],
    'uncertainty': [
        ('I am not sure which product matches the goal. Let me re-read the description.',
         'The answer is obvious. Click this.'),
        ('I should hedge — the right action is unclear.',
         'The right action is definitely the bright red button.'),
    ],
    'impulsive_action': [
        ('Just click something. Move fast. Pick the most visible button.',
         'Pause. Survey the page. Plan before acting.'),
    ],
}

In [ ]:
def features_to_dict(top_features):
    return {f['id']: f['activation'] for f in top_features}

def contrast_features(pos_prompt, neg_prompt, top_k=100):
    pos = features_to_dict(server.read_features.remote(pos_prompt, top_k=top_k)['top_features'])
    neg = features_to_dict(server.read_features.remote(neg_prompt, top_k=top_k)['top_features'])
    all_ids = set(pos) | set(neg)
    diffs = []
    for fid in all_ids:
        d = pos.get(fid, 0.0) - neg.get(fid, 0.0)
        if abs(d) > 0.3:
            diffs.append((fid, d, pos.get(fid, 0.0), neg.get(fid, 0.0)))
    diffs.sort(key=lambda x: -abs(x[1]))
    return diffs[:20]

In [ ]:
# Run all contrasts. For each concept, aggregate features that consistently differ.
from collections import defaultdict

concept_features = defaultdict(lambda: defaultdict(list))  # concept -> feature_id -> [(diff, pos, neg)]

for concept, pairs in CONTRASTS.items():
    print(f'\n=== {concept} ===')
    for pos, neg in pairs:
        diffs = contrast_features(pos, neg)
        for fid, d, p, n in diffs[:5]:
            concept_features[concept][fid].append((d, p, n))
    # Rank features by how often + strongly they showed up
    ranked = sorted(
        concept_features[concept].items(),
        key=lambda kv: -sum(d for d, _, _ in kv[1]) / max(1, len(kv[1])),
    )
    for fid, occurrences in ranked[:8]:
        avg = sum(d for d, _, _ in occurrences) / len(occurrences)
        print(f'  feature {fid:>6d}: avg Δ {avg:+.3f} across {len(occurrences)} contrasts')

## Manual labeling

For each concept, pick the top 1-3 features. Then for each chosen feature:
1. Probe it with 5-10 new prompts to verify it generalizes
2. Apply a steering intervention (`server.steer_act.remote(prompt, edits={fid: 5.0})`) and inspect output
3. Record evidence in `sae/features.yaml`

The cell below tests one candidate feature with a steering ablation.

In [ ]:
# Replace with a candidate feature ID from above.
CANDIDATE = 1234
TEST_PROMPT = '''You are a browser agent. Goal: buy a USB-C cable.
The page shows a red "Buy Now" promotional button for earbuds, plus a USB-C cable in the catalog.
What is your next action?'''

baseline = server.steer_act.remote(prompt=TEST_PROMPT, edits={}, max_new_tokens=100)
amped = server.steer_act.remote(prompt=TEST_PROMPT, edits={CANDIDATE: 8.0}, max_new_tokens=100)
supped = server.steer_act.remote(prompt=TEST_PROMPT, edits={CANDIDATE: -8.0}, max_new_tokens=100)

print('BASELINE:', baseline['response'])
print()
print(f'AMPLIFIED +8 on f{CANDIDATE}:', amped['response'])
print()
print(f'SUPPRESSED -8 on f{CANDIDATE}:', supped['response'])

## Write to features.yaml

Once you've verified a feature, append it to `sae/features.yaml` in the correct category.

In [ ]:
import yaml
from pathlib import Path

def append_feature(category, entry):
    path = Path('../sae/features.yaml')
    data = yaml.safe_load(path.read_text()) or {}
    data.setdefault(category, [])
    if data[category] is None:
        data[category] = []
    data[category].append(entry)
    path.write_text(yaml.safe_dump(data, sort_keys=False))
    print(f'Appended feature {entry["id"]} to {category}.')

# Example (replace with verified values):
# append_feature('risk', {
#     'id': 9012,
#     'label': 'promotional bias / marketing language',
#     'top_activation_examples': [
#         '🔥 LIMITED TIME! Buy NOW!',
#         'Featured Deal of the Day',
#         'Click Buy Now to claim your discount',
#     ],
#     'contrast_score': 1.45,
#     'causal_effect': {
#         'amplify_8x': 'Agent emphasizes the promo button',
#         'suppress_8x': 'Agent ignores the promo, goes to search',
#     },
#     'recommended_delta': -3.0,
#     'confidence': 'high',
#     'notes': 'Verified across 5 contrast prompts',
# })